# Part 0: does the data need composition?

Frozen before the run. Edit only to add what was learned, never to move the prediction after seeing results.

**Question.** Which questions can't be answered closed-book or from any single gold paragraph? Every later result is reported on the full set and on this subset.

**Probes** (B1 text, thinking off): closed book (B0); each gold paragraph with the other gold paragraphs removed and every distractor kept; all paragraphs (B1). A question is labeled only when every probe that applies to it has a row.

**Label.** A question *requires composition* if closed-book is wrong AND every single-gold probe is wrong. The label never depends on the all-paragraphs run being right, because that would select questions the ceiling arm already wins and inflate every gap.

**Prediction.**
- Toy hop1/hop2: about 100% of questions require composition (by construction).
- MuSiQue: a clear majority require it, but not all. In the pilot, the step-2 paragraph alone answered about half as often as both together.
- HotpotQA bridge: a minority require it (Min et al. 2019).
- HotpotQA comparison: the closed-book yes/no rate sits near the majority-class rate.

**Decision rule.**
- If the HotpotQA bridge requires-composition subset is under 30% of bridge questions, HotpotQA is demoted to a secondary set.
- If any toy hop≥1 cell has more than 5% answerable from one shard, the generator leaks and has to be fixed before H1.

**Pilot (Qwen2.5-7B, MuSiQue 2-step, n = 200, K0 channel, EM):** step-1 worker alone 14, step-2 worker alone 30, both 51, single 59. HotpotQA 3B P-split bridge: flat at 34–38 across every arm.

In [ ]:
# Settings. Headless full run: see README "Running".
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SMOKE = True        # True: 2 questions per cell, to read raw outputs first. False: the full design.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from latent_dispatch import analysis, arms, data
from latent_dispatch.model import Model
from latent_dispatch.results import load_results, results_path, run
from latent_dispatch.score import contains_answer, normalize

## Questions

The probes read text in one context, so placement only sets paragraph order: one split placement per question, seed 0.

In [ ]:
qs = (data.load("toy", seed=0, placement="AB")
      + data.load("musique", seed=0, placement=["AB", "AAB", "AABB"])
      + data.load("hotpotqa", seed=0, placement="AB"))
if SMOKE:
    qs = data.per_cell(qs, 2)
print(len(qs), "questions")

# b1_only_hopK keeps only hop K's gold paragraph (distractors stay); it runs only on questions that have one
PROBES = {f"b1_only_hop{h + 1}": h for h in range(4)}
ARMS = {"b0": (arms.b0, {}), "b1": (arms.b1, {}),
        **{name: (arms.b1, {"keep_gold_hops": [hop]}) for name, hop in PROBES.items()}}

## Run

`b0` and `b1` are the same arms H1 uses, so their rows are shared. Probe `b1_only_hopK` keeps only hop K's gold paragraph (distractors stay) and runs only on questions that have a hop-K gold paragraph.

In [ ]:
model = Model.load(MODEL, dtype=DTYPE)
run(model, qs, "b0", arms.b0)
run(model, qs, "b1", arms.b1)
for name, hop in PROBES.items():
    has_hop = [q for q in qs if any(g["hop"] == hop for g in q["gold"])]
    run(model, has_hop, name, arms.b1, keep_gold_hops=[hop])

In [ ]:
df = load_results(qs, ARMS, MODEL, DTYPE)
if SMOKE:
    analysis.peek(df)
analysis.accuracy(df, by=("dataset", "qtype"))

## Requires-composition labels

In [ ]:
w = analysis.wide(df, "em", list(ARMS))
labels, rows = {}, []
for q in qs:
    probes = ["b0"] + [f"b1_only_hop{g['hop'] + 1}" for g in q["gold"]]
    r = w.loc[q["id"]] if q["id"] in w.index else None
    if r is None or any(pd.isna(r.get(p)) for p in probes):     # never label from a partial set of probes
        rows.append({"dataset": q["dataset"], "qtype": q["qtype"], "labeled": False, "requires": None})
        continue
    requires = all(r[p] == 0 for p in probes)
    labels[f"{q['dataset']}:{q['source_id']}"] = requires
    rows.append({"dataset": q["dataset"], "qtype": q["qtype"], "labeled": True, "requires": requires})

lab = pd.DataFrame(rows)
summary = lab[lab.labeled].groupby(["dataset", "qtype"]).requires.agg(n="size", requires="sum")
summary["percent"] = (100 * summary.requires / summary.n).round(1)
print("not labeled (some probe rows missing):", int((~lab.labeled).sum()))
summary

In [ ]:
comp = df[(df.arm == "b0") & (df.qtype == "comparison")]
yes_no = comp[comp.answers.map(lambda a: normalize(a[0]) in ("yes", "no"))]
if len(yes_no):
    yes = yes_no.answers.map(lambda a: normalize(a[0]) == "yes").mean()
    print(f"HotpotQA comparison yes/no: n={len(yes_no)}, majority-class rate {100 * max(yes, 1 - yes):.1f}, "
          f"closed-book EM {100 * yes_no.em.mean():.1f}; entity-answer comparisons: n={len(comp) - len(yes_no)}")

In [ ]:
import json
out = results_path(MODEL, DTYPE, "x").parent / "part0_requires_composition.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(labels, indent=0, sort_keys=True))
print(f"wrote {out} ({len(labels)} questions)")

## Findings

_(fill in after the full run)_